In [1]:
import os
import hashlib

import numpy as np
import pandas as pd
from dotenv import load_dotenv

from hash import generar_hash

In [2]:
load_dotenv()

SALT = os.getenv("HASH_SALT")
if not SALT:
    raise RuntimeError("No se encontró la variable de entorno HASH_SALT")

# Postgres
PG_HOST = os.getenv("PG_HOST")
PG_PORT = os.getenv("PG_PORT", "5432")
PG_DB   = os.getenv("PG_DB")
PG_USER = os.getenv("PG_USER")
PG_PASSWORD = os.getenv("PG_PASSWORD")

# MySQL
MY_HOST = os.getenv("MY_HOST")
MY_PORT = os.getenv("MY_PORT", "3306")
MY_DB   = os.getenv("MY_DB")
MY_USER = os.getenv("MY_USER")
MY_PASSWORD = os.getenv("MY_PASSWORD")


### Limpieza CSV

In [3]:
np.random.seed(42)

df = pd.read_csv("data/df_general_errores.csv")

# Eliminar filas totalmente vacías
df = df.dropna(how="all")

# Limpiar nombre y apellido
for col in ["nombre", "apellido"]:
    if col in df.columns:
        df[col] = (
            df[col]
            .astype(str)
            .replace({"nan": np.nan, "NaN": np.nan})
            .str.strip()
        )

# Mantener solo registros con nombre y apellido válidos
if {"nombre", "apellido"}.issubset(df.columns):
    mask_valid = df["nombre"].notna() & df["apellido"].notna()
    mask_valid &= df["nombre"] != ""
    mask_valid &= df["apellido"] != ""
    df = df[mask_valid].copy()

# Quitar duplicados globales
df = df.drop_duplicates().reset_index(drop=True)

# Definir columnas numéricas (solo si existen)
numeric_cols = ["longitud", "latitud", "distancia", "semestre", "anio", "edad"]
numeric_cols = [c for c in numeric_cols if c in df.columns]

# El resto se consideran texto
text_cols = [c for c in df.columns if c not in numeric_cols]

# --------- Tratamiento de numéricas: outliers + imputación media ---------

for col in numeric_cols:
    col_num = pd.to_numeric(df[col], errors="coerce")

    not_na = col_num.dropna()
    if len(not_na) >= 5:
        q1 = not_na.quantile(0.25)
        q3 = not_na.quantile(0.75)
        iqr = q3 - q1

        lower = q1 - 1.5 * iqr
        upper = q3 + 1.5 * iqr

        outliers = (col_num < lower) | (col_num > upper)
        col_num[outliers] = np.nan

    mean_val = col_num.mean()
    col_num = col_num.fillna(mean_val)

    df[col] = col_num

# Redondeo para variables que deben ser enteras
for col in ["semestre", "anio", "edad"]:
    if col in df.columns:
        df[col] = df[col].round().astype(int)

# --------- Tratamiento de texto: reemplazo de valores corruptos ---------

invalid_tokens = {
    "###", "!ERROR!", "###CORRUPT###", "@@@", "<NULL>",
    "INVALID", "BROKEN", "{BAD}", "<script>",
    "ERR", "???", "BAD", "∞∞∞", "±§¶", "ERR!!", "$$$",
    "<NULL>", "C0RRUPT3D", "[DAMAGED]",
    "~~~~", "XXX", "BAD"
}

def is_invalid_text(v):
    if pd.isna(v):
        return True
    s = str(v).strip()
    if s == "":
        return True
    if s in invalid_tokens:
        return True
    if len(s) > 80:
        return True
    if set(s) <= set("#@!<>={}[]±§¶∞") and len(s) <= 10:
        return True
    return False

for col in text_cols:
    serie = df[col].astype(object)

    valid_mask = ~serie.map(is_invalid_text)
    invalid_mask = ~valid_mask

    # Si todo es inválido, no tocamos la columna
    if valid_mask.sum() == 0:
        df[col] = serie
        continue

    valid_vals = serie[valid_mask].astype(str)
    dist = valid_vals.value_counts(normalize=True)

    valores = dist.index.to_numpy()
    probs = dist.values

    n_invalid = invalid_mask.sum()
    if n_invalid > 0:
        relleno = np.random.choice(valores, size=n_invalid, p=probs)
        serie.loc[invalid_mask] = relleno

    df[col] = serie

# Quitar duplicados por nombre + apellido
if {"nombre", "apellido"}.issubset(df.columns):
    df = df.drop_duplicates(subset=["nombre", "apellido"]).reset_index(drop=True)

df_general = df.reset_index(drop=True)



In [4]:
# Generar identificador único
def generar_hash(nombre, apellido):
    base = f"{nombre}_{apellido}"
    return hashlib.md5(base.encode("utf-8")).hexdigest()

df_general["identifier"] = df_general.apply(
    lambda row: generar_hash(row["nombre"], row["apellido"]),
    axis=1
)

### Limpieza xlsx

In [5]:

df = pd.read_excel("data/df_bienestar_errores.xlsx")
print(df.columns)

np.random.seed(42)

# Eliminar filas totalmente vacías
df = df.dropna(how="all")

# Limpiar nombre y apellido
for col in ["nombre", "apellido"]:
    if col in df.columns:
        df[col] = (
            df[col]
            .astype(str)
            .replace({"nan": np.nan, "NaN": np.nan})
            .str.strip()
        )

# Mantener solo registros con nombre y apellido válidos
if {"nombre", "apellido"}.issubset(df.columns):
    mask_valid = df["nombre"].notna() & df["apellido"].notna()
    mask_valid &= df["nombre"] != ""
    mask_valid &= df["apellido"] != ""
    df = df[mask_valid].copy()

# Quitar duplicados por nombre + apellido
df = df.drop_duplicates(subset=["nombre", "apellido"]).reset_index(drop=True)

# Columnas numéricas (solo si existen)
numeric_cols = [
    "distancia",
    "promedio_académico",
    "viajes_origen",
    "indice_bienestar",
    "distancia_universidad",
    "horas_sueño",
    "tiempo_traslado_min",
]
numeric_cols = [c for c in numeric_cols if c in df.columns]

# El resto se consideran texto
text_cols = [c for c in df.columns if c not in numeric_cols]

# --- Tratamiento numérico: outliers + media ---

for col in numeric_cols:
    col_num = pd.to_numeric(df[col], errors="coerce")

    not_na = col_num.dropna()
    if len(not_na) >= 5:
        q1 = not_na.quantile(0.25)
        q3 = not_na.quantile(0.75)
        iqr = q3 - q1

        lower = q1 - 1.5 * iqr
        upper = q3 + 1.5 * iqr

        outliers = (col_num < lower) | (col_num > upper)
        col_num[outliers] = np.nan

    mean_val = col_num.mean()
    col_num = col_num.fillna(mean_val)

    df[col] = col_num

# Redondear viajes_origen a entero
if "viajes_origen" in df.columns:
    df["viajes_origen"] = df["viajes_origen"].round().astype(int)

# --- Reutilizamos is_invalid_text del código anterior ---

def is_invalid_text(v):
    if pd.isna(v):
        return True
    s = str(v).strip()
    if s == "":
        return True
    invalid_tokens = {
        "###", "!ERROR!", "###CORRUPT###", "@@@", "<NULL>",
        "INVALID", "BROKEN", "{BAD}", "<script>",
        "ERR", "???", "BAD", "∞∞∞", "±§¶", "ERR!!", "$$$",
        "<NULL>", "C0RRUPT3D", "[DAMAGED]",
        "~~~~", "XXX", "BAD"
    }
    if s in invalid_tokens:
        return True
    if len(s) > 80:
        return True
    if set(s) <= set("#@!<>={}[]±§¶∞") and len(s) <= 10:
        return True
    return False

# --- Tratamiento de texto: reemplazo de valores corruptos ---

for col in text_cols:
    serie = df[col].astype(object)

    valid_mask = ~serie.map(is_invalid_text)
    invalid_mask = ~valid_mask

    if valid_mask.sum() == 0:
        df[col] = serie
        continue

    valid_vals = serie[valid_mask].astype(str)
    dist = valid_vals.value_counts(normalize=True)

    valores = dist.index.to_numpy()
    probs = dist.values

    n_invalid = invalid_mask.sum()
    if n_invalid > 0:
        relleno = np.random.choice(valores, size=n_invalid, p=probs)
        serie.loc[invalid_mask] = relleno

    df[col] = serie

# Quitar duplicados por nombre + apellido otra vez por si se generaron
df = df.drop_duplicates(subset=["nombre", "apellido"]).reset_index(drop=True)

# (Opcional) generar el mismo identifier para poder hacer joins
df["identifier"] = df.apply(
    lambda row: generar_hash(row["nombre"], row["apellido"]),
    axis=1
)

df_bienestar = df.reset_index(drop=True)


Index(['nombre', 'apellido', 'ciudad', 'provincia', 'distancia',
       'promedio_académico', 'viajes_origen', 'indice_bienestar',
       'origen_bienestar', 'distancia_universidad', 'horas_sueño',
       'tiempo_traslado_min'],
      dtype='object')


In [6]:

from sqlalchemy import create_engine
# MySQL
mysql_engine = create_engine(
    f"mysql+pymysql://{MY_USER}:{MY_PASSWORD}@{MY_HOST}:{MY_PORT}/{MY_DB}"
)

# Postgres
pg_engine = create_engine(
    f"postgresql+psycopg2://{PG_USER}:{PG_PASSWORD}@{PG_HOST}:{PG_PORT}/{PG_DB}"
)


# Estudiante desde MySQL
df_mysql_estudiante = pd.read_sql("SELECT * FROM estudiante", mysql_engine)
df_mysql_estudiante = df_mysql_estudiante.drop_duplicates(subset=["nombre", "apellido"])



In [34]:


# ---- dim_universidad ----
dim_universidad = (
    df_general[["universidad", "carrera"]]
    .drop_duplicates()
    .sort_values(["universidad", "carrera"])
    .reset_index(drop=True)
)
dim_universidad["id_universidad"] = np.arange(1, len(dim_universidad) + 1)

# ---- dim_periodo ----
dim_periodo = (
    df_general[["semestre", "anio", "periodo"]]
    .drop_duplicates()
    .sort_values(["semestre", "anio", "periodo"])
    .reset_index(drop=True)
)
dim_periodo["id_periodo"] = np.arange(1, len(dim_periodo) + 1)


# ---- dim_estudiante ----
# Usamos el identifier de df_general y cruzamos con MySQL (becado)

dim_estudiante = (
    df_general.merge(
        df_mysql_estudiante[["nombre", "apellido", "es_becado"]],
        on=["nombre", "apellido"],
        how="inner"
    )
)

dim_estudiante = (
    dim_estudiante[["identifier", "edad", "genero", "modalidad", "es_becado"]]
    .drop_duplicates()
    .rename(columns={
        "identifier": "id_estudiante",
        "es_becado": "becado"
    })
    .reset_index(drop=True)
)

# ---- dim_origen ----
dim_origen = (
    df_general[["ciudad", "provincia"]]
    .drop_duplicates()
    .sort_values(["ciudad", "provincia"])
    .reset_index(drop=True)
)
dim_origen["id_origen"] = np.arange(1, len(dim_origen) + 1)

# ---- dim_fuente_ingreso ----
df_mysql_tipo_ingreso = pd.read_sql("SELECT * FROM tipo_ingreso", mysql_engine)

dim_fuente_ingreso = (
    df_mysql_tipo_ingreso[["id_tipo_ingreso", "nombre"]]
    .drop_duplicates()
    .sort_values(["id_tipo_ingreso"])
    .reset_index(drop=True)
    .rename(columns={
        "id_tipo_ingreso": "id_fuente_ingresos",
        "nombre": "tipo_fuente"
    })
)



In [29]:


query_ingresos_full = """
SELECT
    e.nombre,
    e.apellido,
    sum_all.ingreso_total,
    top_tipo.id_tipo_ingreso,
    ti.nombre AS tipo_ingreso_principal,
    top_tipo.total_ingreso AS ingreso_principal
FROM
    -- Suma total por estudiante
    (
        SELECT
            i.id_estudiante,
            SUM(i.ingreso_cantidad) AS ingreso_total
        FROM ingreso i
        GROUP BY i.id_estudiante
    ) AS sum_all
INNER JOIN estudiante e
    ON e.id_estudiante = sum_all.id_estudiante
INNER JOIN (
    -- Suma por tipo y estudiante, y nos quedamos con el tipo que más aporta
    SELECT
        i.id_estudiante,
        i.id_tipo_ingreso,
        SUM(i.ingreso_cantidad) AS total_ingreso,
        ROW_NUMBER() OVER (
            PARTITION BY i.id_estudiante
            ORDER BY SUM(i.ingreso_cantidad) DESC
        ) AS rn
    FROM ingreso i
    GROUP BY i.id_estudiante, i.id_tipo_ingreso
) AS top_tipo
    ON sum_all.id_estudiante = top_tipo.id_estudiante
    AND top_tipo.rn = 1
INNER JOIN tipo_ingreso ti
    ON ti.id_tipo_ingreso = top_tipo.id_tipo_ingreso;
"""

df_ingresos_full = pd.read_sql(query_ingresos_full, mysql_engine)


# Egresos totales por estudiante (Postgres)
query_egresos_totales = """
    SELECT 
        e.nombre,
        e.apellido,
        SUM(i.egreso_cantidad) AS egreso_total
    FROM gastos i
    JOIN estudiante e ON i.id_estudiante = e.id_estudiante
    GROUP BY e.nombre, e.apellido
"""
df_egresos_totales = pd.read_sql(query_egresos_totales, pg_engine)





In [32]:
df_economia = df_ingresos_full.merge(
    df_egresos_totales,          # tiene egreso_total
    on=["nombre", "apellido"],
    how="inner"                  # como pediste, inner join
)

df_economia["balance_neto"] = (
    df_economia["ingreso_total"] - df_economia["egreso_total"]
)


In [35]:
df_economia_fk = df_economia.merge(
    dim_fuente_ingreso[["id_fuente_ingresos", "tipo_fuente"]],
    left_on="id_tipo_ingreso",
    right_on="id_fuente_ingresos",
    how="inner"
)

In [37]:
general_keys = (
    df_general[
        [
            "nombre",
            "apellido",
            "universidad",
            "carrera",
            "semestre",
            "anio",
            "periodo",
            "ciudad",
            "provincia",
        ]
    ]
    .drop_duplicates()
)
# mapa estudiante: identifier -> id_estudiante
map_estudiante = (
    df_general[["identifier", "nombre", "apellido"]]
    .drop_duplicates()
    .rename(columns={"identifier": "id_estudiante"})
)


In [38]:

fact = (
    df_economia_fk
    # universidad / periodo / origen
    .merge(general_keys, on=["nombre", "apellido"], how="inner")
    # estudiante
    .merge(map_estudiante, on=["nombre", "apellido"], how="inner")
    # universidad -> id_universidad
    .merge(
        dim_universidad[["id_universidad", "universidad", "carrera"]],
        on=["universidad", "carrera"],
        how="inner",
    )
    # periodo -> id_periodo
    .merge(
        dim_periodo[["id_periodo", "semestre", "anio", "periodo"]],
        on=["semestre", "anio", "periodo"],
        how="inner",
    )
    # origen -> id_origen
    .merge(
        dim_origen[["id_origen", "ciudad", "provincia"]],
        on=["ciudad", "provincia"],
        how="inner",
    )
)

# ================== 4. Quedarse solo con FKs + medidas ==================

fact_economia = fact[
    [
        "id_estudiante",
        "id_periodo",
        "id_universidad",
        "id_origen",
        "id_fuente_ingresos",
        "ingreso_total",
        "egreso_total",
        "balance_neto",
    ]
].drop_duplicates().copy()


fact_economia = fact_economia.rename(
    columns={
        "ingreso_total": "ingresos_totales",
        "egreso_total": "egresos_totales",
    }
)

In [39]:

ids_est = fact_economia["id_estudiante"].unique()
ids_per = fact_economia["id_periodo"].unique()
ids_uni = fact_economia["id_universidad"].unique()
ids_ori = fact_economia["id_origen"].unique()
ids_fuente = fact_economia["id_fuente_ingresos"].unique()

# ================== Filtrar dimensiones ==================

dim_estudiante_clean = dim_estudiante[
    dim_estudiante["id_estudiante"].isin(ids_est)
].reset_index(drop=True)

dim_periodo_clean = dim_periodo[
    dim_periodo["id_periodo"].isin(ids_per)
].reset_index(drop=True)

dim_universidad_clean = dim_universidad[
    dim_universidad["id_universidad"].isin(ids_uni)
].reset_index(drop=True)

dim_origen_clean = dim_origen[
    dim_origen["id_origen"].isin(ids_ori)
].reset_index(drop=True)

dim_fuente_ingreso_clean = dim_fuente_ingreso[
    dim_fuente_ingreso["id_fuente_ingresos"].isin(ids_fuente)
].reset_index(drop=True)


In [41]:
dim_estudiante=dim_estudiante_clean
dim_periodo=dim_periodo_clean
dim_universidad=dim_universidad_clean
dim_origen=dim_origen_clean
dim_fuente_ingreso=dim_fuente_ingreso_clean

# Cargar datos

In [43]:
import os
import clickhouse_connect
from dotenv import load_dotenv

load_dotenv()

CH_HOST = os.getenv("CH_HOST", "localhost")
CH_PORT = int(os.getenv("CH_PORT", "8123"))
CH_USER = os.getenv("CH_USER", "admin")
CH_PASSWORD = os.getenv("CH_PASSWORD", "admin")

CH_DB = "dm_economia"

client = clickhouse_connect.get_client(
    host=CH_HOST,
    port=CH_PORT,
    username=CH_USER,
    password=CH_PASSWORD,
)


In [44]:
def insert_df(client, table, df):
    cols = list(df.columns)
    data = [tuple(row) for row in df.itertuples(index=False, name=None)]
    client.insert(table, data, column_names=cols)

CH_DM_ECO = "dm_economia"

insert_df(client, f"{CH_DM_ECO}.dim_estudiante", dim_estudiante_clean)
insert_df(client, f"{CH_DM_ECO}.dim_universidad", dim_universidad_clean)
insert_df(client, f"{CH_DM_ECO}.dim_periodo", dim_periodo_clean)
insert_df(client, f"{CH_DM_ECO}.dim_origen", dim_origen_clean)
insert_df(client, f"{CH_DM_ECO}.dim_fuente_ingreso", dim_fuente_ingreso_clean)



In [45]:

# ==== FACT ====
insert_df(client, f"{CH_DM_ECO}.fact_economia", fact_economia)

### Resultado

Después de este proceso, al final quedo un 67% de los datos perfectos para un análisis